In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


# Load data
class_train_data = pd.read_csv('../Data/AppML_InitialProject_train.csv')
class_test_data = pd.read_csv('../Data/AppML_InitialProject_test_classification.csv')

# Prepare features and target
target = class_train_data['p_Truth_isElectron']
variables = class_train_data.drop(columns=['p_Truth_isElectron', 'p_Truth_Energy'])

# Split into train/validation for calculating importance on unseen data
X_train, X_val, y_train, y_val = train_test_split(
    variables, target, test_size=0.2, random_state=42, stratify=target
)

# Train a classifier
print("\n" + "="*60)
print("Training Random Forest classifier...")
print("="*60)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Evaluate on validation set
train_score = model.score(X_train, y_train)
val_score = model.score(X_val, y_val)
print(f"\nTrain accuracy: {train_score:.4f}")
print(f"Validation accuracy: {val_score:.4f}")

# Calculate permutation importance on validation set
print("\n" + "="*60)
print("Calculating Permutation Importance...")
print("="*60)

perm_importance = permutation_importance(
    model, X_val, y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

# Create results dataframe
importance_df = pd.DataFrame({
    'feature': variables.columns,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("\nPermutation Importance Results:")
print(importance_df.to_string(index=False))

# Bar plot
top_n = 15
top_features = importance_df.head(top_n)
colors = plt.cm.viridis(np.linspace(0, 1, len(top_features)))
plt.figure(figsize=(12, 8))
plt.barh(top_features['feature'], top_features['importance'], 
         xerr=top_features['std'], color=colors, capsize=5)
plt.xlabel('Permutation Importance', fontsize=11)
plt.title(f'Top {top_n} Feature Importance\n(Permutation-based)', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()

In [ ]:
#Retrain with top features
top_feature_names = top_features['feature'].tolist()
X_train_top = X_train[top_feature_names]
X_val_top = X_val[top_feature_names]

model_top = RandomForestClassifier(
    n_estimators=500,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)
model_top.fit(X_train_top, y_train)
train_score_top = model_top.score(X_train_top, y_train)
val_score_top = model_top.score(X_val_top, y_val)
print(f"\nTrain accuracy with top features: {train_score_top:.4f}")
print(f"Validation accuracy with top features: {val_score_top:.4f}")



Train accuracy with top features: 0.9861
Validation accuracy with top features: 0.9657


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Hyperparameter tuning on the reduced 15-feature set
# This matches the feature limit and avoids tuning a larger model than you can submit.
param_distributions = {
    'n_estimators': randint(100, 800),
    'max_depth': [None] + list(range(3, 21)),
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=25,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train_top, y_train)

print(f"Best CV score: {random_search.best_score_:.4f}")
print(f"Best parameters: {random_search.best_params_}")

best_model = random_search.best_estimator_
val_score_tuned = best_model.score(X_val_top, y_val)
print(f"Validation accuracy after tuning: {val_score_tuned:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Define hyperparameter grid for random search
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(5, 30),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf': randint(1, 10),
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False],
    'criterion': ['gini', 'entropy']
}

# Perform RandomizedSearchCV on top features
print("\n" + "="*60)
print("Hyperparameter Tuning with RandomizedSearchCV...")
print("="*60)

random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist,
    n_iter=50,  # Number of parameter combinations to try
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train_top, y_train)

# Display results
print("\n" + "="*60)
print("RandomizedSearchCV Results")
print("="*60)
print(f"\nBest parameters: {random_search.best_params_}")
print(f"Best cross-validation score: {random_search.best_score_:.4f}")

# Evaluate best model on validation set
best_model = random_search.best_estimator_
train_score_tuned = best_model.score(X_train_top, y_train)
val_score_tuned = best_model.score(X_val_top, y_val)

print(f"\nTrain accuracy with tuned hyperparameters: {train_score_tuned:.4f}")
print(f"Validation accuracy with tuned hyperparameters: {val_score_tuned:.4f}")

# Compare models
print("\n" + "="*60)
print("Model Comparison")
print("="*60)
print(f"Baseline (initial model): {val_score:.4f}")
print(f"With top 15 features: {val_score_top:.4f}")
print(f"With tuned hyperparameters: {val_score_tuned:.4f}")